# Notebook 3 — Clasificadores RNN (LSTM · BiLSTM · GRU · SimpleRNN)

**Proyecto:** Sistema Web para Apoyo en Decisiones de Inversión usando IA  
**Curso:** Introducción al Desarrollo de Software (iDeSo) — Semana 11  
**Universidad:** UNMSM · FISI  
**Docente:** Prof. Mg. Ing. Ernesto D. Cancho-Rodríguez, MBA (GWU)

---

## Objetivo
Implementar y comparar **4 arquitecturas de redes neuronales recurrentes** para clasificación
binaria de tendencias bursátiles (BUY / SELL):

| Modelo     | Arquitectura                                               | Épocas | Batch |
|------------|------------------------------------------------------------|--------|-------|
| LSTM       | LSTM(260) → Dropout(0.2) → LSTM(130) → Dense(1, sigmoid) | 80     | 64    |
| BiLSTM     | BiLSTM(200) → Dropout(0.3) → BiLSTM(100) → Dense(1, sigmoid) | 80 | 64    |
| GRU        | GRU(280) → GRU(140) → Dense(1, sigmoid)                  | 80     | 64    |
| SimpleRNN  | RNN(180) → RNN(90) → Dense(1, sigmoid)                   | 80     | 64    |

**Frontend destino:** `ernesto_investing_rnns_clasificador.html` (Anexo C.3)  
**Salida:** `datos_rnns.json` con estructura exacta que el HTML consume.

---
**Referencia doctoral:** Inspirado en los notebooks multi-arquitectura del profesor:  
`26_05_12_MultiArqui_{1FSM,2VOLCAN,3ABX,4BVN,5BHP}v2_v2E4C2(Opti)(BDA2026_1)_UpToMod11.ipynb`

## Celda 0 — Instalación de Librerías

In [1]:
# ─────────────────────────────────────────────────────────────────
# CELDA 0: Instalación de librerías requeridas
# Ejecutar sólo una vez al inicio de la sesión de Colab
# ─────────────────────────────────────────────────────────────────
!pip install yfinance ta --quiet
print('✅ Librerías instaladas correctamente.')

  Preparing metadata (setup.py) ... done
✅ Librerías instaladas correctamente.


## Celda 1 — Importaciones y Semilla Global

In [2]:
# ─────────────────────────────────────────────────────────────────
# CELDA 1: Importaciones y configuración de semilla de reproducibilidad
# ─────────────────────────────────────────────────────────────────
import os
import json
import warnings
import numpy as np
import pandas as pd
import yfinance as yf
import ta
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score
)
from datetime import datetime, timedelta

# Suprimir avisos no críticos
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# ── Semilla global para reproducibilidad ──────────────────────────
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

print(f'✅ TensorFlow {tf.__version__} | NumPy {np.__version__} | Pandas {pd.__version__}')
print(f'📌 Semilla global fijada: SEED={SEED}')

# Verificar GPU disponible
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'🚀 GPU disponible: {gpus[0].name}')
else:
    print('⚠️  Sin GPU — entrenamiento en CPU (más lento pero funcional)')

✅ TensorFlow 2.20.0 | NumPy 2.0.2 | Pandas 2.2.2
📌 Semilla global fijada: SEED=42
🚀 GPU disponible: /physical_device:GPU:0


## Celda 2 — Configuración de Tickers y Parámetros

In [3]:
# ─────────────────────────────────────────────────────────────────
# CELDA 2: Configuración central del notebook
# Todos los parámetros del proyecto en un solo lugar
# ─────────────────────────────────────────────────────────────────

# ── 5 Activos financieros del estudio (mineras con operaciones en Perú) ──
TICKERS = {
    'FSM':         'Fortuna Silver Mines Inc.',
    'VOLCABC1.LM': 'Volcan Compañía Minera S.A.A.',
    'ABX.TO':      'Barrick Gold Corporation',
    'BVN':         'Compañía de Minas Buenaventura S.A.A.',
    'BHP':         'BHP Billiton Limited'
}

# ── Período temporal: últimos 3 años (más datos = mejor entrenamiento) ──
FECHA_FIN   = datetime.today().strftime('%Y-%m-%d')
FECHA_INICIO = (datetime.today() - timedelta(days=3*365)).strftime('%Y-%m-%d')

# ── Parámetros de secuencia temporal ──
VENTANA = 20          # días de historia para cada secuencia de entrada

# ── Parámetros de entrenamiento (según especificaciones del profesor) ──
EPOCAS    = 80
BATCH     = 64
LR        = 0.001     # tasa de aprendizaje Adam
SPLIT     = 0.80      # 80% entrenamiento / 20% prueba

# ── Arquitecturas de referencia (según especificaciones) ──
MODELOS_CONFIG = [
    {
        'id':    'LSTM',
        'arq':  'LSTM(260)→Drop(0.2)→LSTM(130)→Dense(1)',
        'color': '#1565c0'
    },
    {
        'id':    'BiLSTM',
        'arq':  'BiLSTM(200)→Drop(0.3)→BiLSTM(100)→Dense(1)',
        'color': '#6a1b9a'
    },
    {
        'id':    'GRU',
        'arq':  'GRU(280)→GRU(140)→Dense(1)',
        'color': '#e65100'
    },
    {
        'id':    'SimpleRNN',
        'arq':  'RNN(180)→RNN(90)→Dense(1)',
        'color': '#00838f'
    }
]

# ── Nombre del archivo JSON de salida ──
ARCHIVO_JSON = 'datos_rnns.json'

print('📋 Configuración del Notebook 3:')
print(f'   Tickers : {list(TICKERS.keys())}')
print(f'   Período : {FECHA_INICIO} → {FECHA_FIN}')
print(f'   Ventana : {VENTANA} días')
print(f'   Épocas  : {EPOCAS} | Batch: {BATCH} | LR: {LR}')
print(f'   Split   : {int(SPLIT*100)}% train / {int((1-SPLIT)*100)}% test')
print(f'   Modelos : {[m["id"] for m in MODELOS_CONFIG]}')
print(f'   Salida  : {ARCHIVO_JSON}')

📋 Configuración del Notebook 3:
   Tickers : ['FSM', 'VOLCABC1.LM', 'ABX.TO', 'BVN', 'BHP']
   Período : 2023-06-17 → 2026-06-16
   Ventana : 20 días
   Épocas  : 80 | Batch: 64 | LR: 0.001
   Split   : 80% train / 19% test
   Modelos : ['LSTM', 'BiLSTM', 'GRU', 'SimpleRNN']
   Salida  : datos_rnns.json


## Celda 3 — Descarga de Datos OHLCV con yfinance

In [4]:
# ─────────────────────────────────────────────────────────────────
# CELDA 3: Descarga de datos reales con yfinance
# Incluye manejo de errores para tickers que no descarguen
# NOTA: Nunca se inventan datos — sólo datos reales de Yahoo Finance
# ─────────────────────────────────────────────────────────────────

def descargar_datos(ticker: str, inicio: str, fin: str) -> pd.DataFrame | None:
    """
    Descarga datos OHLCV de Yahoo Finance para un ticker dado.

    Parámetros:
        ticker : símbolo bursátil (ej: 'BVN')
        inicio : fecha de inicio en formato 'YYYY-MM-DD'
        fin    : fecha de fin en formato 'YYYY-MM-DD'

    Retorna:
        DataFrame con columnas OHLCV o None si la descarga falla.
    """
    try:
        df = yf.download(ticker, start=inicio, end=fin, progress=False, auto_adjust=True)

        # Verificar que hay datos suficientes
        if df.empty or len(df) < VENTANA * 3:
            print(f'  ⚠️  {ticker}: datos insuficientes ({len(df)} filas). Se omite.')
            return None

        # Aplanar columnas MultiIndex que genera yfinance con auto_adjust=True
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        # Renombrar columnas a español para claridad
        df = df.rename(columns={
            'Open': 'apertura', 'High': 'maximo', 'Low': 'minimo',
            'Close': 'cierre', 'Volume': 'volumen'
        })

        # Eliminar filas con valores nulos en precio de cierre
        df = df.dropna(subset=['cierre'])
        df.index = pd.to_datetime(df.index)

        print(f'  ✅ {ticker}: {len(df)} sesiones descargadas '
              f'({df.index[0].date()} → {df.index[-1].date()})')
        return df

    except Exception as e:
        print(f'  ❌ {ticker}: error al descargar → {e}')
        return None


# ── Descargar todos los tickers ────────────────────────────────────
print('📡 Descargando datos de Yahoo Finance...')
print(f'   Período: {FECHA_INICIO} → {FECHA_FIN}\n')

datos_crudos = {}   # diccionario: ticker → DataFrame

for ticker in TICKERS:
    df = descargar_datos(ticker, FECHA_INICIO, FECHA_FIN)
    if df is not None:
        datos_crudos[ticker] = df

print(f'\n📊 Tickers disponibles para entrenamiento: {list(datos_crudos.keys())}')
print(f'⚠️  Tickers excluidos: {[t for t in TICKERS if t not in datos_crudos]}')

📡 Descargando datos de Yahoo Finance...
   Período: 2023-06-17 → 2026-06-16

  ✅ FSM: 750 sesiones descargadas (2023-06-20 → 2026-06-15)
  ✅ VOLCABC1.LM: 741 sesiones descargadas (2023-06-20 → 2026-06-15)
  ✅ ABX.TO: 751 sesiones descargadas (2023-06-19 → 2026-06-15)
  ✅ BVN: 750 sesiones descargadas (2023-06-20 → 2026-06-15)
  ✅ BHP: 750 sesiones descargadas (2023-06-20 → 2026-06-15)

📊 Tickers disponibles para entrenamiento: ['FSM', 'VOLCABC1.LM', 'ABX.TO', 'BVN', 'BHP']
⚠️  Tickers excluidos: []


## Celda 4 — Ingeniería de Características e Indicadores Técnicos

In [5]:
# ─────────────────────────────────────────────────────────────────
# CELDA 4: Cálculo de indicadores técnicos y variable objetivo
# Usando la librería 'ta' (Technical Analysis)
# Variable objetivo binaria: 1=BUY (tendencia alcista), 0=SELL (bajista)
# ─────────────────────────────────────────────────────────────────

def calcular_indicadores(df: pd.DataFrame) -> pd.DataFrame | None:
    """
    Agrega indicadores técnicos al DataFrame de precios OHLCV.

    Indicadores calculados:
        - SMA-20, SMA-50 (medias móviles simples)
        - EMA-12, EMA-26 (medias móviles exponenciales)
        - RSI-14 (Índice de Fuerza Relativa)
        - MACD, señal MACD, histograma MACD
        - Bandas de Bollinger (superior, media, inferior)
        - ATR-14 (Average True Range — volatilidad)
        - Retorno diario y retorno logarítmico
        - Variable objetivo: Trend (1=BUY, 0=SELL)
    """
    df = df.copy()

    # ── Medias Móviles Simples ────────────────────────────────────
    df['sma_20'] = ta.trend.sma_indicator(df['cierre'], window=20)
    df['sma_50'] = ta.trend.sma_indicator(df['cierre'], window=50)

    # ── Medias Móviles Exponenciales ─────────────────────────────
    df['ema_12'] = ta.trend.ema_indicator(df['cierre'], window=12)
    df['ema_26'] = ta.trend.ema_indicator(df['cierre'], window=26)

    # ── RSI ───────────────────────────────────────────────────────
    df['rsi_14'] = ta.momentum.rsi(df['cierre'], window=14)

    # ── MACD ──────────────────────────────────────────────────────
    macd_obj = ta.trend.MACD(df['cierre'], window_slow=26, window_fast=12, window_sign=9)
    df['macd']        = macd_obj.macd()
    df['macd_signal'] = macd_obj.macd_signal()
    df['macd_hist']   = macd_obj.macd_diff()

    # ── Bandas de Bollinger ───────────────────────────────────────
    bb = ta.volatility.BollingerBands(df['cierre'], window=20, window_dev=2)
    df['bb_superior'] = bb.bollinger_hband()
    df['bb_media']    = bb.bollinger_mavg()
    df['bb_inferior'] = bb.bollinger_lband()

    # ── ATR (volatilidad) ─────────────────────────────────────────
    df['atr_14'] = ta.volatility.average_true_range(
        df['maximo'], df['minimo'], df['cierre'], window=14
    )

    # ── Retornos ──────────────────────────────────────────────────
    df['retorno']      = df['cierre'].pct_change()
    df['retorno_log']  = np.log(df['cierre'] / df['cierre'].shift(1))

    # ── Variable objetivo: tendencia del día siguiente ────────────
    # 1 = BUY (precio de mañana > precio de hoy)
    # 0 = SELL (precio de mañana <= precio de hoy)
    df['trend'] = (df['cierre'].shift(-1) > df['cierre']).astype(int)

    # Eliminar filas con NaN (causadas por ventanas de indicadores)
    df = df.dropna()

    # Eliminar la última fila (trend del último día es desconocido)
    df = df.iloc[:-1]

    if len(df) < VENTANA * 2:
        print(f'  ⚠️  Datos insuficientes tras calcular indicadores: {len(df)} filas')
        return None

    return df


# ── Aplicar a todos los tickers disponibles ───────────────────────
print('⚙️  Calculando indicadores técnicos...\n')

datos_procesados = {}   # ticker → DataFrame con indicadores

# Columnas que se usarán como features de entrada al modelo
FEATURES = [
    'cierre', 'apertura', 'maximo', 'minimo', 'volumen',
    'sma_20', 'sma_50', 'ema_12', 'ema_26',
    'rsi_14', 'macd', 'macd_signal', 'macd_hist',
    'bb_superior', 'bb_media', 'bb_inferior',
    'atr_14', 'retorno', 'retorno_log'
]

for ticker, df_raw in datos_crudos.items():
    df_proc = calcular_indicadores(df_raw)
    if df_proc is not None:
        datos_procesados[ticker] = df_proc
        balance = df_proc['trend'].value_counts(normalize=True)
        print(f'  ✅ {ticker}: {len(df_proc)} sesiones | '
              f'BUY={balance.get(1,0):.1%} | SELL={balance.get(0,0):.1%}')

print(f'\n🎯 Features de entrada ({len(FEATURES)}): {FEATURES}')

⚙️  Calculando indicadores técnicos...

  ✅ FSM: 700 sesiones | BUY=49.9% | SELL=50.1%
  ✅ VOLCABC1.LM: 691 sesiones | BUY=43.3% | SELL=56.7%
  ✅ ABX.TO: 701 sesiones | BUY=54.9% | SELL=45.1%
  ✅ BVN: 700 sesiones | BUY=52.4% | SELL=47.6%
  ✅ BHP: 700 sesiones | BUY=52.9% | SELL=47.1%

🎯 Features de entrada (19): ['cierre', 'apertura', 'maximo', 'minimo', 'volumen', 'sma_20', 'sma_50', 'ema_12', 'ema_26', 'rsi_14', 'macd', 'macd_signal', 'macd_hist', 'bb_superior', 'bb_media', 'bb_inferior', 'atr_14', 'retorno', 'retorno_log']


## Celda 5 — Funciones de Preprocesamiento y Creación de Secuencias

In [6]:
# ─────────────────────────────────────────────────────────────────
# CELDA 5: Preprocesamiento de datos para modelos RNN
# - Normalización con MinMaxScaler
# - Creación de secuencias temporales de longitud VENTANA
# - División temporal train/test (sin data leakage)
# ─────────────────────────────────────────────────────────────────

def crear_secuencias(X: np.ndarray, y: np.ndarray,
                     ventana: int) -> tuple[np.ndarray, np.ndarray]:
    """
    Transforma arrays planos en secuencias temporales para RNN.

    Para cada posición i >= ventana:
        X_seq[i] = X[i-ventana : i]   (shape: ventana × n_features)
        y_seq[i] = y[i]               (etiqueta del día i)

    Parámetros:
        X      : array de features (n_dias × n_features)
        y      : array de etiquetas (n_dias,)
        ventana: número de días históricos en cada secuencia

    Retorna:
        X_seq : (n_secuencias, ventana, n_features)
        y_seq : (n_secuencias,)
    """
    X_seq, y_seq = [], []
    for i in range(ventana, len(X)):
        X_seq.append(X[i - ventana:i])
        y_seq.append(y[i])
    return np.array(X_seq, dtype=np.float32), np.array(y_seq, dtype=np.float32)


def preparar_datos(df: pd.DataFrame):
    """
    Pipeline completo de preprocesamiento para un ticker:
    1. Extraer features y etiquetas
    2. División temporal train/test (SPLIT% / (1-SPLIT)%)
    3. Normalización: MinMaxScaler ajustado SÓLO en train
    4. Crear secuencias temporales

    Retorna diccionario con arrays y scaler para reproducibilidad.
    """
    X_raw = df[FEATURES].values
    y_raw = df['trend'].values

    # ── División temporal sin mezclar (preservar orden cronológico) ──
    corte = int(len(X_raw) * SPLIT)
    X_train_raw, X_test_raw = X_raw[:corte], X_raw[corte:]
    y_train_raw, y_test_raw = y_raw[:corte], y_raw[corte:]

    # ── Normalización: scaler ajustado sólo en train (evitar data leakage) ──
    scaler = MinMaxScaler(feature_range=(0, 1))
    X_train_norm = scaler.fit_transform(X_train_raw)
    X_test_norm  = scaler.transform(X_test_raw)    # sólo transform, sin fit

    # ── Crear secuencias temporales ───────────────────────────────
    X_train, y_train = crear_secuencias(X_train_norm, y_train_raw, VENTANA)
    X_test,  y_test  = crear_secuencias(X_test_norm,  y_test_raw,  VENTANA)

    return {
        'X_train': X_train, 'y_train': y_train,
        'X_test':  X_test,  'y_test':  y_test,
        'scaler':  scaler,
        'n_features': X_train.shape[2]
    }


# ── Preprocesar todos los tickers ─────────────────────────────────
print('🔧 Preprocesando datos...\n')

datos_ml = {}   # ticker → dict con arrays de train/test

for ticker, df in datos_procesados.items():
    prep = preparar_datos(df)
    datos_ml[ticker] = prep
    print(f'  {ticker}:')
    print(f'    Train → X:{prep["X_train"].shape} | y:{prep["y_train"].shape}')
    print(f'    Test  → X:{prep["X_test"].shape}  | y:{prep["y_test"].shape}')

print(f'\n✅ Preprocesamiento completado.')
print(f'   Forma de entrada: (batch, {VENTANA} días, {len(FEATURES)} features)')

🔧 Preprocesando datos...

  FSM:
    Train → X:(540, 20, 19) | y:(540,)
    Test  → X:(120, 20, 19)  | y:(120,)
  VOLCABC1.LM:
    Train → X:(532, 20, 19) | y:(532,)
    Test  → X:(119, 20, 19)  | y:(119,)
  ABX.TO:
    Train → X:(540, 20, 19) | y:(540,)
    Test  → X:(121, 20, 19)  | y:(121,)
  BVN:
    Train → X:(540, 20, 19) | y:(540,)
    Test  → X:(120, 20, 19)  | y:(120,)
  BHP:
    Train → X:(540, 20, 19) | y:(540,)
    Test  → X:(120, 20, 19)  | y:(120,)

✅ Preprocesamiento completado.
   Forma de entrada: (batch, 20 días, 19 features)


## Celda 6 — Construcción de los 4 Modelos RNN

In [7]:
# ─────────────────────────────────────────────────────────────────
# CELDA 6: Definición de las 4 arquitecturas RNN
# Cada función construye y compila un modelo Keras según las
# especificaciones exactas del Prof. Cancho-Rodríguez
# ─────────────────────────────────────────────────────────────────

def construir_lstm(n_features: int) -> keras.Model:
    """
    Modelo LSTM — 2 capas LSTM con dropout.
    Arquitectura: LSTM(260, return_seq=True) → Dropout(0.2)
                  → LSTM(130) → Dense(1, sigmoid)
    """
    modelo = keras.Sequential([
        keras.Input(shape=(VENTANA, n_features)),
        layers.LSTM(260, return_sequences=True, name='lstm_1'),
        layers.Dropout(0.2, name='dropout_1'),
        layers.LSTM(130, return_sequences=False, name='lstm_2'),
        layers.Dense(1, activation='sigmoid', name='salida')
    ], name='LSTM_Clasificador')

    modelo.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LR),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return modelo


def construir_bilstm(n_features: int) -> keras.Model:
    """
    Modelo BiLSTM — 2 capas LSTM bidireccionales con dropout.
    Arquitectura: Bidirectional(LSTM(200, return_seq=True)) → Dropout(0.3)
                  → Bidirectional(LSTM(100)) → Dense(1, sigmoid)
    La capa bidireccional procesa la secuencia en ambas direcciones
    (hacia adelante y hacia atrás), duplicando la capacidad representacional.
    """
    modelo = keras.Sequential([
        keras.Input(shape=(VENTANA, n_features)),
        layers.Bidirectional(
            layers.LSTM(200, return_sequences=True), name='bilstm_1'
        ),
        layers.Dropout(0.3, name='dropout_1'),
        layers.Bidirectional(
            layers.LSTM(100, return_sequences=False), name='bilstm_2'
        ),
        layers.Dense(1, activation='sigmoid', name='salida')
    ], name='BiLSTM_Clasificador')

    modelo.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LR),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return modelo


def construir_gru(n_features: int) -> keras.Model:
    """
    Modelo GRU — 2 capas GRU sin dropout (diseño intencional).
    Arquitectura: GRU(280, return_seq=True) → GRU(140)
                  → Dense(1, sigmoid)
    GRU tiene menos parámetros que LSTM pero similar capacidad:
    usa compuertas de actualización y reinicio en lugar de
    compuertas de entrada, olvido y salida del LSTM.
    """
    modelo = keras.Sequential([
        keras.Input(shape=(VENTANA, n_features)),
        layers.GRU(280, return_sequences=True, name='gru_1'),
        layers.GRU(140, return_sequences=False, name='gru_2'),
        layers.Dense(1, activation='sigmoid', name='salida')
    ], name='GRU_Clasificador')

    modelo.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LR),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return modelo


def construir_simple_rnn(n_features: int) -> keras.Model:
    """
    Modelo SimpleRNN — baseline de comparación.
    Arquitectura: SimpleRNN(180, return_seq=True) → SimpleRNN(90)
                  → Dense(1, sigmoid)
    SimpleRNN es la arquitectura más básica: sólo mantiene un
    estado oculto sin compuertas. Sirve como línea base para
    evaluar si las arquitecturas con compuertas (LSTM, GRU)
    realmente superan al modelo más simple.
    """
    modelo = keras.Sequential([
        keras.Input(shape=(VENTANA, n_features)),
        layers.SimpleRNN(180, return_sequences=True, name='rnn_1'),
        layers.SimpleRNN(90, return_sequences=False, name='rnn_2'),
        layers.Dense(1, activation='sigmoid', name='salida')
    ], name='SimpleRNN_Clasificador')

    modelo.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LR),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return modelo


# Mapa de constructor por ID de modelo
CONSTRUCTORES = {
    'LSTM':      construir_lstm,
    'BiLSTM':    construir_bilstm,
    'GRU':       construir_gru,
    'SimpleRNN': construir_simple_rnn
}

# ── Mostrar resumen de los modelos para un ticker de ejemplo ──────
n_feat_ejemplo = len(FEATURES)
print('📐 Arquitecturas de los modelos RNN:\n')
for cfg in MODELOS_CONFIG:
    modelo_demo = CONSTRUCTORES[cfg['id']](n_feat_ejemplo)
    params = modelo_demo.count_params()
    print(f'  [{cfg["id"]}] {cfg["arq"]}')
    print(f'         Parámetros totales: {params:,}')
    keras.backend.clear_session()   # liberar memoria entre demos
print('\n✅ Arquitecturas definidas correctamente.')

📐 Arquitecturas de los modelos RNN:

  [LSTM] LSTM(260)→Drop(0.2)→LSTM(130)→Dense(1)
         Parámetros totales: 494,651
  [BiLSTM] BiLSTM(200)→Drop(0.3)→BiLSTM(100)→Dense(1)
         Parámetros totales: 753,001
  [GRU] GRU(280)→GRU(140)→Dense(1)
         Parámetros totales: 430,221
  [SimpleRNN] RNN(180)→RNN(90)→Dense(1)
         Parámetros totales: 60,481

✅ Arquitecturas definidas correctamente.


## Celda 7 — Entrenamiento de los 4 Modelos para cada Ticker

In [8]:
# ─────────────────────────────────────────────────────────────────
# CELDA 7: Entrenamiento completo de los 4 modelos × 5 tickers
# ─────────────────────────────────────────────────────────────────
#
# NOTA IMPORTANTE: Entrenar 4 modelos × N tickers × 80 épocas
# puede tomar entre 20-40 minutos en CPU y ~5-10 min en GPU.
# Se recomienda activar la GPU en Colab:
# Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU
# ─────────────────────────────────────────────────────────────────

# Diccionario principal de resultados: ticker → modelo → resultados
resultados = {}   # estructura: {ticker: {modelo_id: {...}}}

# Callback: detención temprana si el val_loss no mejora en 15 épocas
early_stopping = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True,
    verbose=0
)

for ticker in datos_ml:
    print(f'\n{'='*60}')
    print(f'🏦 TICKER: {ticker} — {TICKERS[ticker]}')
    print(f'{'='*60}')

    prep       = datos_ml[ticker]
    X_train    = prep['X_train']
    y_train    = prep['y_train']
    X_test     = prep['X_test']
    y_test     = prep['y_test']
    n_features = prep['n_features']

    resultados[ticker] = {}

    for cfg in MODELOS_CONFIG:
        modelo_id = cfg['id']
        print(f'\n  ── Entrenando {modelo_id} ──────────────────────────')

        # Fijar semilla antes de construir cada modelo
        tf.random.set_seed(SEED)
        np.random.seed(SEED)

        # Construir modelo
        modelo = CONSTRUCTORES[modelo_id](n_features)

        # Entrenar
        hist = modelo.fit(
            X_train, y_train,
            epochs=EPOCAS,
            batch_size=BATCH,
            validation_split=0.15,   # 15% de train para validación
            callbacks=[early_stopping],
            verbose=0                # silencioso (sin barra de progreso)
        )

        epocas_reales = len(hist.history['accuracy'])
        print(f'     Épocas completadas: {epocas_reales}/{EPOCAS}')

        # ── Predicciones en conjunto de prueba ───────────────────
        y_prob  = modelo.predict(X_test, verbose=0).flatten()
        y_pred  = (y_prob >= 0.5).astype(int)
        confianza_ultima = float(y_prob[-1])
        senal_ultima     = 'BUY' if y_pred[-1] == 1 else 'SELL'

        # ── Métricas de evaluación ───────────────────────────────
        acc  = accuracy_score(y_test,  y_pred)
        prec = precision_score(y_test, y_pred, zero_division=0)
        rec  = recall_score(y_test,   y_pred, zero_division=0)
        f1   = f1_score(y_test,       y_pred, zero_division=0)

        print(f'     Accuracy : {acc:.4f} ({acc*100:.1f}%)')
        print(f'     Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f}')
        print(f'     Señal última sesión: {senal_ultima} (conf: {confianza_ultima:.2f})')

        # ── Historial de accuracy por época (para gráfico del HTML) ──
        acc_hist = [round(float(a), 4) for a in hist.history['accuracy']]
        # Rellenar con el último valor si early stopping cortó antes de EPOCAS
        if len(acc_hist) < EPOCAS:
            acc_hist += [acc_hist[-1]] * (EPOCAS - len(acc_hist))

        # ── Señales diarias sobre el conjunto de prueba ──────────
        senales_pred = ['BUY' if p == 1 else 'SELL' for p in y_pred]

        # Guardar resultados del modelo
        resultados[ticker][modelo_id] = {
            'acc':     round(acc, 4),
            'prec':    round(prec, 4),
            'rec':     round(rec, 4),
            'f1':      round(f1, 4),
            'senal':   senal_ultima,
            'conf':    round(confianza_ultima, 2),
            'accHist': acc_hist,
            'senales': senales_pred
        }

        # Liberar memoria del modelo (importante en Colab con GPU)
        keras.backend.clear_session()
        del modelo

print('\n' + '='*60)
print('✅ ENTRENAMIENTO COMPLETADO')
print('='*60)


🏦 TICKER: FSM — Fortuna Silver Mines Inc.

  ── Entrenando LSTM ──────────────────────────
     Épocas completadas: 16/80
     Accuracy : 0.5167 (51.7%)
     Precision: 0.5167 | Recall: 1.0000 | F1: 0.6813
     Señal última sesión: BUY (conf: 0.52)

  ── Entrenando BiLSTM ──────────────────────────
     Épocas completadas: 16/80
     Accuracy : 0.5167 (51.7%)
     Precision: 0.5167 | Recall: 1.0000 | F1: 0.6813
     Señal última sesión: BUY (conf: 0.54)

  ── Entrenando GRU ──────────────────────────


     Épocas completadas: 15/80


     Accuracy : 0.5167 (51.7%)
     Precision: 0.5167 | Recall: 1.0000 | F1: 0.6813
     Señal última sesión: BUY (conf: 0.55)

  ── Entrenando SimpleRNN ──────────────────────────
     Épocas completadas: 15/80
     Accuracy : 0.5250 (52.5%)
     Precision: 0.5214 | Recall: 0.9839 | F1: 0.6816
     Señal última sesión: BUY (conf: 0.54)

🏦 TICKER: VOLCABC1.LM — Volcan Compañía Minera S.A.A.

  ── Entrenando LSTM ──────────────────────────
     Épocas completadas: 15/80
     Accuracy : 0.4874 (48.7%)
     Precision: 0.0000 | Recall: 0.0000 | F1: 0.0000
     Señal última sesión: SELL (conf: 0.36)

  ── Entrenando BiLSTM ──────────────────────────
     Épocas completadas: 15/80
     Accuracy : 0.4874 (48.7%)
     Precision: 0.0000 | Recall: 0.0000 | F1: 0.0000
     Señal última sesión: SELL (conf: 0.39)

  ── Entrenando GRU ──────────────────────────
     Épocas completadas: 15/80
     Accuracy : 0.4874 (48.7%)
     Precision: 0.0000 | Recall: 0.0000 | F1: 0.0000
     Señal última sesión:

## Celda 8 — Resumen Comparativo de Resultados

In [9]:
# ─────────────────────────────────────────────────────────────────
# CELDA 8: Tabla comparativa de métricas por ticker y modelo
# Análisis del mejor modelo y análisis de la bidireccionalidad
# ─────────────────────────────────────────────────────────────────

print('📊 RESUMEN COMPARATIVO DE MÉTRICAS\n')
print(f'  {"Ticker":<14} {"Modelo":<12} {"Accuracy":>9} {"Precision":>10} {"Recall":>8} {"F1":>8} {"Señal":<6}')
print('  ' + '-' * 70)

for ticker in resultados:
    mejor_acc  = max(resultados[ticker].values(), key=lambda x: x['acc'])
    mejor_mod  = [m for m, v in resultados[ticker].items() if v['acc'] == mejor_acc['acc']][0]

    for modelo_id in ['LSTM', 'BiLSTM', 'GRU', 'SimpleRNN']:
        if modelo_id not in resultados[ticker]:
            continue
        r = resultados[ticker][modelo_id]
        marca = ' ⭐' if modelo_id == mejor_mod else ''
        print(f'  {ticker:<14} {modelo_id + marca:<14} '
              f'{r["acc"]*100:>7.2f}%  {r["prec"]*100:>8.2f}%  '
              f'{r["rec"]*100:>6.2f}%  {r["f1"]*100:>6.2f}%  {r["senal"]}')
    print()

# ── Análisis: ¿Las compuertas mejoran al SimpleRNN? ──────────────
print('\n🔍 ANÁLISIS: Arquitecturas con compuertas vs. SimpleRNN')
print('-' * 55)
for ticker in resultados:
    if 'SimpleRNN' not in resultados[ticker]:
        continue
    acc_rnn = resultados[ticker]['SimpleRNN']['acc']
    for mid in ['LSTM', 'BiLSTM', 'GRU']:
        if mid not in resultados[ticker]:
            continue
        acc_adv = resultados[ticker][mid]['acc']
        delta = acc_adv - acc_rnn
        signo = '▲' if delta >= 0 else '▼'
        print(f'  {ticker} | {mid} vs SimpleRNN: {signo} {abs(delta)*100:.2f}% puntos')

# ── Análisis: ¿BiLSTM supera a LSTM unidireccional? ─────────────
print('\n🔍 ANÁLISIS: BiLSTM vs. LSTM (bidireccionalidad)')
print('-' * 55)
for ticker in resultados:
    if 'LSTM' not in resultados[ticker] or 'BiLSTM' not in resultados[ticker]:
        continue
    acc_lstm   = resultados[ticker]['LSTM']['acc']
    acc_bilstm = resultados[ticker]['BiLSTM']['acc']
    delta = acc_bilstm - acc_lstm
    signo = '▲' if delta >= 0 else '▼'
    concl = 'BiLSTM SUPERA a LSTM' if delta > 0 else 'LSTM supera a BiLSTM'
    print(f'  {ticker}: {signo} {abs(delta)*100:.2f}% puntos → {concl}')

📊 RESUMEN COMPARATIVO DE MÉTRICAS

  Ticker         Modelo        Accuracy  Precision   Recall       F1 Señal 
  ----------------------------------------------------------------------
  FSM            LSTM             51.67%     51.67%  100.00%   68.13%  BUY
  FSM            BiLSTM           51.67%     51.67%  100.00%   68.13%  BUY
  FSM            GRU              51.67%     51.67%  100.00%   68.13%  BUY
  FSM            SimpleRNN ⭐      52.50%     52.14%   98.39%   68.16%  BUY

  VOLCABC1.LM    LSTM ⭐           48.74%      0.00%    0.00%    0.00%  SELL
  VOLCABC1.LM    BiLSTM           48.74%      0.00%    0.00%    0.00%  SELL
  VOLCABC1.LM    GRU              48.74%      0.00%    0.00%    0.00%  SELL
  VOLCABC1.LM    SimpleRNN        48.74%      0.00%    0.00%    0.00%  SELL

  ABX.TO         LSTM             44.63%      0.00%    0.00%    0.00%  SELL
  ABX.TO         BiLSTM           44.63%      0.00%    0.00%    0.00%  SELL
  ABX.TO         GRU              44.63%      0.00%    0.0

## Celda 9 — Construcción del JSON para el Frontend

In [10]:
# ─────────────────────────────────────────────────────────────────
# CELDA 9: Construcción de la estructura JSON exacta que consume
# el archivo ernesto_investing_rnns_clasificador.html
#
# La función generarDatos() del HTML espera exactamente:
#   datos[ticker] = {
#     fechas   : [str, ...],
#     precios  : [float, ...],
#     modelos  : [
#       {
#         id      : str,        # 'LSTM' | 'BiLSTM' | 'GRU' | 'SimpleRNN'
#         arq     : str,        # descripción de la arquitectura
#         color   : str,        # color hex del modelo
#         acc     : float,      # accuracy 0-1
#         prec    : float,      # precision 0-1
#         rec     : float,      # recall 0-1
#         f1      : float,      # F1-score 0-1
#         accHist : [float×80], # accuracy por época (80 valores)
#         senales : [str, ...], # 'BUY' | 'SELL' por cada fecha de precios
#         senal   : str,        # señal del último día
#         conf    : float       # confianza de la señal 0-1
#       }, ...
#     ]
#   }
# ─────────────────────────────────────────────────────────────────

def construir_json(datos_procesados: dict, resultados: dict,
                   datos_ml: dict) -> dict:
    """
    Construye el diccionario JSON completo para todos los tickers.

    El JSON une:
      - Fechas y precios reales de Yahoo Finance
      - Métricas, señales e historial de cada modelo RNN
    """
    salida = {}

    # Mapa de configuración por id de modelo
    cfg_map = {c['id']: c for c in MODELOS_CONFIG}

    for ticker, df in datos_procesados.items():
        if ticker not in resultados:
            continue

        # ── Precios del conjunto de PRUEBA (últimos datos disponibles) ──
        # Usar el mismo corte que preparar_datos() usó
        corte = int(len(df) * SPLIT)
        df_test = df.iloc[corte:]

        # Alinear con las secuencias creadas (se pierden VENTANA filas al inicio)
        df_senales = df_test.iloc[VENTANA:].copy()

        fechas  = [str(d.date()) for d in df_senales.index]
        precios = [round(float(v), 4) for v in df_senales['cierre'].values]

        # ── Construir lista de modelos ────────────────────────────
        lista_modelos = []
        for cfg in MODELOS_CONFIG:
            mid = cfg['id']
            if mid not in resultados[ticker]:
                continue

            r = resultados[ticker][mid]

            # Ajustar señales para que coincidan exactamente con fechas/precios
            senales_ajustadas = r['senales'][:len(fechas)]
            # Si hay menos señales que fechas, rellenar con la última señal
            if len(senales_ajustadas) < len(fechas):
                senales_ajustadas += [r['senal']] * (len(fechas) - len(senales_ajustadas))

            lista_modelos.append({
                'id':      mid,
                'arq':     cfg['arq'],
                'color':   cfg['color'],
                'acc':     r['acc'],
                'prec':    r['prec'],
                'rec':     r['rec'],
                'f1':      r['f1'],
                'accHist': r['accHist'],    # siempre 80 valores
                'senales': senales_ajustadas,
                'senal':   r['senal'],
                'conf':    r['conf']
            })

        salida[ticker] = {
            'ticker':        ticker,
            'nombre':        TICKERS[ticker],
            'fecha_generado': datetime.today().strftime('%Y-%m-%d %H:%M:%S'),
            'periodo':        f'{FECHA_INICIO} → {FECHA_FIN}',
            'ventana':        VENTANA,
            'epocas':         EPOCAS,
            'fechas':         fechas,
            'precios':        precios,
            'modelos':        lista_modelos
        }

    return salida


# ── Construir el JSON ─────────────────────────────────────────────
print('🏗️  Construyendo estructura JSON...')
datos_json = construir_json(datos_procesados, resultados, datos_ml)

# ── Validación de la estructura ───────────────────────────────────
print('\n🔍 Validando estructura JSON:\n')
for ticker, contenido in datos_json.items():
    n_fechas  = len(contenido['fechas'])
    n_precios = len(contenido['precios'])
    print(f'  {ticker}:')
    print(f'    Fechas/Precios: {n_fechas} / {n_precios}')
    for m in contenido['modelos']:
        n_senales = len(m['senales'])
        n_hist    = len(m['accHist'])
        ok_s = '✅' if n_senales == n_fechas else '❌'
        ok_h = '✅' if n_hist == EPOCAS else '⚠️'
        print(f'    {m["id"]:<12} Señales:{n_senales}{ok_s}  '
              f'accHist:{n_hist}{ok_h}  '
              f'Acc:{m["acc"]*100:.1f}%  Señal:{m["senal"]}({m["conf"]:.0%})')

🏗️  Construyendo estructura JSON...

🔍 Validando estructura JSON:

  FSM:
    Fechas/Precios: 120 / 120
    LSTM         Señales:120✅  accHist:80✅  Acc:51.7%  Señal:BUY(52%)
    BiLSTM       Señales:120✅  accHist:80✅  Acc:51.7%  Señal:BUY(54%)
    GRU          Señales:120✅  accHist:80✅  Acc:51.7%  Señal:BUY(55%)
    SimpleRNN    Señales:120✅  accHist:80✅  Acc:52.5%  Señal:BUY(54%)
  VOLCABC1.LM:
    Fechas/Precios: 119 / 119
    LSTM         Señales:119✅  accHist:80✅  Acc:48.7%  Señal:SELL(36%)
    BiLSTM       Señales:119✅  accHist:80✅  Acc:48.7%  Señal:SELL(39%)
    GRU          Señales:119✅  accHist:80✅  Acc:48.7%  Señal:SELL(39%)
    SimpleRNN    Señales:119✅  accHist:80✅  Acc:48.7%  Señal:SELL(23%)
  ABX.TO:
    Fechas/Precios: 121 / 121
    LSTM         Señales:121✅  accHist:80✅  Acc:44.6%  Señal:SELL(33%)
    BiLSTM       Señales:121✅  accHist:80✅  Acc:44.6%  Señal:SELL(29%)
    GRU          Señales:121✅  accHist:80✅  Acc:44.6%  Señal:SELL(27%)
    SimpleRNN    Señales:121✅  acc

## Celda 10 — Exportación a JSON (Contrato de Datos)

In [11]:
# ─────────────────────────────────────────────────────────────────
# CELDA 10: Exportación final del archivo JSON
# Este archivo es el CONTRATO DE DATOS entre el backend (Python)
# y el frontend (HTML/JS: ernesto_investing_rnns_clasificador.html)
# ─────────────────────────────────────────────────────────────────

# Exportar a archivo JSON con formato legible
with open(ARCHIVO_JSON, 'w', encoding='utf-8') as f:
    json.dump(datos_json, f, ensure_ascii=False, indent=2)

# Calcular tamaño del archivo
import os
tamano_bytes = os.path.getsize(ARCHIVO_JSON)
tamano_kb    = tamano_bytes / 1024

print('='*60)
print('✅ EXPORTACIÓN COMPLETADA')
print('='*60)
print(f'  Archivo : {ARCHIVO_JSON}')
print(f'  Tamaño  : {tamano_kb:.1f} KB ({tamano_bytes:,} bytes)')
print(f'  Tickers : {list(datos_json.keys())}')
print()
print('📋 Estructura del JSON exportado:')
print(json.dumps({
    k: {
        'ticker':   v['ticker'],
        'n_fechas': len(v['fechas']),
        'modelos':  [m['id'] for m in v['modelos']]
    } for k, v in datos_json.items()
}, indent=2, ensure_ascii=False))

# Descargar archivo automáticamente en Google Colab
try:
    from google.colab import files
    files.download(ARCHIVO_JSON)
    print(f'\n⬇️  Archivo {ARCHIVO_JSON} descargado automáticamente.')
except ImportError:
    print(f'\n💡 Para descargar el archivo, ejecute:')
    print(f'   from google.colab import files')
    print(f'   files.download("{ARCHIVO_JSON}")')

print()
print('🌐 Próximo paso — conectar con el frontend:')
print(f'   Copiar {ARCHIVO_JSON} en la misma carpeta que:')
print('   ernesto_investing_rnns_clasificador.html')
print('   y modificar el HTML para usar fetch() en lugar de generarDatos()')

✅ EXPORTACIÓN COMPLETADA
  Archivo : datos_rnns.json
  Tamaño  : 96.7 KB (99,037 bytes)
  Tickers : ['FSM', 'VOLCABC1.LM', 'ABX.TO', 'BVN', 'BHP']

📋 Estructura del JSON exportado:
{
  "FSM": {
    "ticker": "FSM",
    "n_fechas": 120,
    "modelos": [
      "LSTM",
      "BiLSTM",
      "GRU",
      "SimpleRNN"
    ]
  },
  "VOLCABC1.LM": {
    "ticker": "VOLCABC1.LM",
    "n_fechas": 119,
    "modelos": [
      "LSTM",
      "BiLSTM",
      "GRU",
      "SimpleRNN"
    ]
  },
  "ABX.TO": {
    "ticker": "ABX.TO",
    "n_fechas": 121,
    "modelos": [
      "LSTM",
      "BiLSTM",
      "GRU",
      "SimpleRNN"
    ]
  },
  "BVN": {
    "ticker": "BVN",
    "n_fechas": 120,
    "modelos": [
      "LSTM",
      "BiLSTM",
      "GRU",
      "SimpleRNN"
    ]
  },
  "BHP": {
    "ticker": "BHP",
    "n_fechas": 120,
    "modelos": [
      "LSTM",
      "BiLSTM",
      "GRU",
      "SimpleRNN"
    ]
  }
}


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


⬇️  Archivo datos_rnns.json descargado automáticamente.

🌐 Próximo paso — conectar con el frontend:
   Copiar datos_rnns.json en la misma carpeta que:
   ernesto_investing_rnns_clasificador.html
   y modificar el HTML para usar fetch() en lugar de generarDatos()


## Celda 11 — (Opcional) Verificación: Preview del JSON

In [12]:
# ─────────────────────────────────────────────────────────────────
# CELDA 11 (OPCIONAL): Muestra un preview del JSON exportado
# para verificar que la estructura es correcta antes de
# integrarlo con el frontend HTML
# ─────────────────────────────────────────────────────────────────

# Leer el JSON exportado para verificar integridad
with open(ARCHIVO_JSON, 'r', encoding='utf-8') as f:
    verificacion = json.load(f)

# Mostrar preview del primer ticker disponible
primer_ticker = list(verificacion.keys())[0]
datos_ticker  = verificacion[primer_ticker]

print(f'📄 PREVIEW del JSON — Ticker: {primer_ticker}\n')
print(f'  nombre          : {datos_ticker["nombre"]}')
print(f'  fecha_generado  : {datos_ticker["fecha_generado"]}')
print(f'  periodo         : {datos_ticker["periodo"]}')
print(f'  ventana         : {datos_ticker["ventana"]} días')
print(f'  n_fechas        : {len(datos_ticker["fechas"])}')
print(f'  primeras fechas : {datos_ticker["fechas"][:3]}')
print(f'  últimas fechas  : {datos_ticker["fechas"][-3:]}')
print(f'  primeros precios: {datos_ticker["precios"][:3]}')
print(f'  últimos precios : {datos_ticker["precios"][-3:]}')
print()

print('  MODELOS:')
for m in datos_ticker['modelos']:
    print(f'  ┌─ {m["id"]}')
    print(f'  │  arq     : {m["arq"]}')
    print(f'  │  acc     : {m["acc"]*100:.2f}%')
    print(f'  │  prec    : {m["prec"]*100:.2f}%')
    print(f'  │  rec     : {m["rec"]*100:.2f}%')
    print(f'  │  f1      : {m["f1"]*100:.2f}%')
    print(f'  │  senal   : {m["senal"]} ({m["conf"]*100:.0f}% confianza)')
    print(f'  │  accHist : [{m["accHist"][0]:.4f}, ..., {m["accHist"][-1]:.4f}] ({len(m["accHist"])} épocas)')
    print(f'  └  señales : {m["senales"][:3]} ... {m["senales"][-3:]} ({len(m["senales"])} total)')
    print()

print('✅ JSON verificado — estructura correcta para el frontend HTML.')

📄 PREVIEW del JSON — Ticker: FSM

  nombre          : Fortuna Silver Mines Inc.
  fecha_generado  : 2026-06-16 20:52:07
  periodo         : 2023-06-17 → 2026-06-16
  ventana         : 20 días
  n_fechas        : 120
  primeras fechas : ['2025-12-19', '2025-12-22', '2025-12-23']
  últimas fechas  : ['2026-06-10', '2026-06-11', '2026-06-12']
  primeros precios: [9.85, 10.17, 10.18]
  últimos precios : [8.09, 8.61, 8.93]

  MODELOS:
  ┌─ LSTM
  │  arq     : LSTM(260)→Drop(0.2)→LSTM(130)→Dense(1)
  │  acc     : 51.67%
  │  prec    : 51.67%
  │  rec     : 100.00%
  │  f1      : 68.13%
  │  senal   : BUY (52% confianza)
  │  accHist : [0.4880, ..., 0.5643] (80 épocas)
  └  señales : ['BUY', 'BUY', 'BUY'] ... ['BUY', 'BUY', 'BUY'] (120 total)

  ┌─ BiLSTM
  │  arq     : BiLSTM(200)→Drop(0.3)→BiLSTM(100)→Dense(1)
  │  acc     : 51.67%
  │  prec    : 51.67%
  │  rec     : 100.00%
  │  f1      : 68.13%
  │  senal   : BUY (54% confianza)
  │  accHist : [0.4771, ..., 0.5708] (80 épocas)
  └  señal

## Celda 12 — (Opcional) Patch del HTML para consumir el JSON real

In [13]:
# ─────────────────────────────────────────────────────────────────
# CELDA 12 (OPCIONAL): Genera el snippet de código JS que debe
# reemplazar la función generarDatos() del HTML para leer el JSON
# real en lugar de generar datos simulados con Math.random()
#
# Instrucción: copiar este snippet y pegarlo en el HTML,
# reemplazando la función renderizar() original.
# ─────────────────────────────────────────────────────────────────

snippet_js = '''
// ── REEMPLAZAR función generarDatos() por fetch() al JSON real ──
// Colocar datos_rnns.json en la misma carpeta que el HTML

let DATOS_REALES = null;   // caché del JSON descargado

async function cargarDatosReales() {
  if (DATOS_REALES) return DATOS_REALES;   // ya cargado, usar caché
  try {
    const resp = await fetch('datos_rnns.json');   // o URL de la API
    if (!resp.ok) throw new Error(`HTTP ${resp.status}`);
    DATOS_REALES = await resp.json();
    console.log('✅ JSON de datos reales cargado:', Object.keys(DATOS_REALES));
    return DATOS_REALES;
  } catch (err) {
    console.warn('⚠️ No se pudo cargar datos_rnns.json:', err.message);
    console.warn('   Usando datos simulados como fallback.');
    return null;
  }
}

// Reemplazar la función renderizar() original con esta versión async:
async function renderizarConDatosReales() {
  const ticker = document.getElementById('selTicker').value;
  const todos  = await cargarDatosReales();

  let datos;
  if (todos && todos[ticker]) {
    // ── Usar datos reales del JSON ────────────────────────────────
    const d = todos[ticker];
    datos = {
      fechas:  d.fechas,
      precios: d.precios,
      modelos: d.modelos.map(m => ({
        id:      m.id,
        arq:     m.arq,
        color:   m.color,
        acc:     m.acc,
        prec:    m.prec,
        rec:     m.rec,
        f1:      m.f1,
        accHist: m.accHist,
        senales: m.senales,
        senal:   m.senal,
        conf:    m.conf
      }))
    };
    console.log(`🏦 Datos reales cargados para ${ticker}`);
  } else {
    // ── Fallback a datos simulados si el JSON no está disponible ──
    console.warn(`Ticker ${ticker} no encontrado en el JSON. Usando simulados.`);
    datos = generarDatos(ticker);   // función original del HTML
  }

  // ── El resto del código de renderizado NO cambia ────────────────
  // (tarjetas, tabla, radar, gráficos — idéntico al HTML original)
  renderizarConDatos(datos, ticker);
}

// Reemplazar el onclick del botón y la inicialización:
// Cambiar: onclick="renderizar()"
// Por    : onclick="renderizarConDatosReales()"
// Y al final del <script>:
// Cambiar: renderizar();
// Por    : renderizarConDatosReales();
'''

print('📋 SNIPPET JS para conectar el HTML con el JSON real:')
print('='*60)
print(snippet_js)
print('='*60)
print()
print('📁 Archivos necesarios en la misma carpeta:')
print(f'   ├── ernesto_investing_rnns_clasificador.html')
print(f'   └── {ARCHIVO_JSON}')
print()
print('🌐 Para la Fase 2 (FastAPI), el fetch apuntaría a:')
print(f'   fetch("http://TU-NGROK-URL/api/rnns/" + ticker)')

📋 SNIPPET JS para conectar el HTML con el JSON real:

// ── REEMPLAZAR función generarDatos() por fetch() al JSON real ──
// Colocar datos_rnns.json en la misma carpeta que el HTML

let DATOS_REALES = null;   // caché del JSON descargado

async function cargarDatosReales() {
  if (DATOS_REALES) return DATOS_REALES;   // ya cargado, usar caché
  try {
    const resp = await fetch('datos_rnns.json');   // o URL de la API
    if (!resp.ok) throw new Error(`HTTP ${resp.status}`);
    DATOS_REALES = await resp.json();
    console.log('✅ JSON de datos reales cargado:', Object.keys(DATOS_REALES));
    return DATOS_REALES;
  } catch (err) {
    console.warn('⚠️ No se pudo cargar datos_rnns.json:', err.message);
    console.warn('   Usando datos simulados como fallback.');
    return null;
  }
}

// Reemplazar la función renderizar() original con esta versión async:
async function renderizarConDatosReales() {
  const ticker = document.getElementById('selTicker').value;
  const todos  = await ca